In [ ]:
!pip intsall medmnist
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import medmnist
from medmnist import INFO, Evaluator


# loding the dataset(diabatic retinopathy)
data_flag="retinamnist"
info= INFO[data_flag]
n_classses= len(info["label"]) #getting 5 classes represents 5 class of diabatic retinopathic deaseas.
print("Number of classes:", n_classses)
print(info)
DataClass= getattr(medmnist, info["python_class"])

# preprocessing== converting images to matrix and normalizing it

data_transform= transforms.Compose([
    transforms.ToTensor(), transforms.Normalize(mean=[0.5], std=[0.5])
])

# prepearing training and testing data
print("downloding datasets...")
train_data= DataClass(split="train", transform= data_transform, download= True)
test_data= DataClass(split="test", transform= data_transform, download= True)

# loading the data

train_loader= DataLoader(dataset= train_data, batch_size= 16, shuffle= True)
test_loader= DataLoader(dataset= test_data, batch_size= 16, shuffle= False)

print("status...")
print(f"Dataset loaded succesfully")
print(f"Total training images: {len(train_data)}")
print(f"Number of traget classes: {n_classses} {info["label"]}")



# feetching 1 batch to verofy the tensor.
data_iter=iter(train_loader)
images, labels= next(data_iter)
print(f"Image batch shape: {images.shape}")
print(f"Label batch shape: {labels.shape}")

n_qubits= 6 # definfing no. of qubits I will use in our QC.

# starting the CNN framwork for image classification.


class FeatureExtractor(nn.Module):
  def __init__(self):
    super().__init__()
    # convolution block= extract patterns form image
    # RGB image = 3 I/P channels (28x28 pixels)
    self.conv_block= nn.Sequential(
        nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2)
    )
     # Dense layer=hidden layers(compresses down to 4 classical no.)
    self.flatten= nn.Flatten() # flatten layer(1D vector)
    self.dense= nn.Linear(8*14*14, n_qubits)

  def forward(self, x):
    x= self.conv_block(x) #Extract features
    x= self.flatten(x)   # Flatten to 1D vector
    x= self.dense(x)     # compress to 4 feature input
    return x


# initialize my CNN.
cnn= FeatureExtractor()
extract_feat= cnn(images)
print("stattus..")
print(f"original image batch shape:{images.shape},(Batch,Channels, Height, Width)")
print(f"Extracted 4 features(raw) for 1st image only:\n {extract_feat[0].detach().numpy()}")

# building the Quantum Circuit
# building the Torch Layer
# Connecting them both


from pennylane import wires
#setting up the Quantum simulator
dev=qml.device("default.qubit", wires=n_qubits)

# defining the QNODE(QNN)
@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs,weights):
  # Angle Encodding: Mapping the 4 feat. to R_y rotation angles.
  qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
  # Variational Ansatz: entangl + trainanle Rotations
  qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
  # Measurement: <Pauli-z> [-1.0,1.0]
  return [qml.expval(qml.PauliZ(wires=i))
          for i in range(n_qubits)]

# setting up the Pytorch layer
# I choose 2 layers of entanglement (2 layers*4 qubits= 8 trainable angles)
n_layers=4
weight_shapes={"weights": (n_layers, n_qubits)}

#Bridging the Quantm Circuit into a Pytorch Layer
qlayer= qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

#Testing by pushing the 4 feat.as I/P into the Quantum Circuit.
quantum_op= qlayer(extract_feat)

print("status...")
print(f"I/P shape to Quantum Layer: {extract_feat.shape}")
print(f"O/P shape from Quantum Layer: {quantum_op.shape} (batch,Expctation_values)")
print(f"Expectation Values for 1st image:\n {quantum_op[0].detach().numpy()}")

# Building the HYBRID Model
class Retina_hybrid(nn.Module):
  def __init__(self):
    super().__init__()
    self.cnn= FeatureExtractor()
    self.qlayer= qml.qnn.TorchLayer(quantum_circuit, weight_shapes)
    self.fc= nn.Linear(n_qubits, 5)
  def forward(self,x):
    x= self.cnn(x)
    x= self.qlayer(x)
    x= self.fc(x)
    return x
# Initializing
model= Retina_hybrid()

# defining the loss & oprtimizer
loss= nn.CrossEntropyLoss()
optimizer= torch.optim.Adam(model.parameters(), lr=0.001)




#Training Loop
print("status...")
epochs=30
for epoch in range(epochs):
  for i,(batch_images, batch_labels) in enumerate(train_loader):
    batch_labels = batch_labels.squeeze()
    optimizer.zero_grad()
    output= model(batch_images)
    batch_loss= loss(output, batch_labels)

    #triggering parameter-shift rule
    batch_loss.backward()
    optimizer.step()

    if (i+1)%10==0:
      print (f"Epoch [{epoch+1}/{epochs}],Batch [{i+1}/{len(train_loader)}],Loss: {batch_loss.item():.4f}")

print("\nTraining Complete!")


# MODEL EVALUATION

model.eval()
correct=0
total=0

print("status..")
print("testing the Hybrid Model with testing data")

with torch.no_grad():
  for batch_images, batch_labels in test_loader:
    batch_labels= batch_labels.squeeze()

    # Predictions from model
    output= model(batch_images)

    # output is the P distribution across 5 classes
    # taking the class with the highest P value
    _, predicted= torch.max(output.data, 1)

    # counting the predictions that match the actual labels
    total += batch_labels.size(0)
    correct += (predicted== batch_labels).sum().item()

# calculating the accuracy_score
accuracy= 100* correct/total
print(f"\nFinal Accuracy:{accuracy:.2f}%")
print(f"Correctly Predicted {correct} out of {total} test images.")

#saving the trained model

model_path="medmnist_hybrid_vqc_weights.pth"
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")

print(f"Model successfully saved as:{model_path}")


from google.colab import files
files.download('medmnist_hybrid_vqc_weights.pth')

